Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
# MAGIC
# MAGIC import trino
# MAGIC import pandas as pd
# MAGIC
# MAGIC conn = trino.dbapi.connect(
# MAGIC     host="trino-tableau-prod.dp.navi-tech.in",
# MAGIC     port=443,
# MAGIC     user="databricks-user",
# MAGIC     catalog="awsdatacatalog",
# MAGIC     schema="default",
# MAGIC     http_scheme="https",
# MAGIC )
# MAGIC
# MAGIC
# MAGIC query = """
# MAGIC -- SPADE CATEGORY MATCHING RANDOMIZER QUERY (Updated)
# MAGIC -- Returns 25% random sample of SPADE Category Matching V1 data
# MAGIC -- Filters: Completed on B-1 & B-2, >= 3 tickets, includes 'Three -' and 'Two-One' types only
# MAGIC -- Uses updated DA base query for alignment and accurate dates
# MAGIC SELECT * FROM (
# MAGIC     WITH base_table AS (
# MAGIC         SELECT
# MAGIC             a.*,
# MAGIC             b.td AS "Time Taken (Seconds)",
# MAGIC             c.average_time AS "Avg Project Time",
# MAGIC             c.median_time AS "Median Project Time",
# MAGIC             project_wise_daily_ticket_counts,
# MAGIC             project_wise_daily_time,
# MAGIC             ROUND((project_wise_daily_ticket_counts / project_wise_daily_time) * c.average_time, 2) AS "Project level Agent Productivity"
# MAGIC         FROM (
# MAGIC             SELECT
# MAGIC                 annotated_at AS "Annotation Date",
# MAGIC                 CONCAT(correlation_id, '__', CAST(set AS VARCHAR)) AS "Correlation Id",
# MAGIC                 ticket_id AS "Ticket Id",
# MAGIC                 project_type AS "Project Type",
# MAGIC                 u.name AS "Agent Name",
# MAGIC                 Option_type AS "Option Type",
# MAGIC                 type AS "Type",
# MAGIC                 u.email_id
# MAGIC             FROM corp_analytics.data_annotation_projects_p1_tbl p1
# MAGIC             LEFT JOIN mjolnir_service.users u ON u.email_id = p1.email_id
# MAGIC             WHERE project_type NOT IN ('App Tagging', 'Address Tagging')
# MAGIC             UNION ALL
# MAGIC             SELECT
# MAGIC                 annotated_at AS "Annotation Date",
# MAGIC                 correlation_id AS "Correlation Id",
# MAGIC                 ticket_id AS "Ticket Id",
# MAGIC                 'Address annotation metaaddress' AS "Project Type",
# MAGIC                 u.name AS "Agent Name",
# MAGIC                 option_type AS "Option Type",
# MAGIC                 type AS "Type",
# MAGIC                 u.email_id
# MAGIC             FROM corp_analytics_databricks.data_annotation_meta_address ma
# MAGIC             LEFT JOIN mjolnir_service.users u ON u.email_id = ma.email_id
# MAGIC             UNION ALL
# MAGIC             SELECT
# MAGIC                 annotated_at AS "Annotation Date",
# MAGIC                 correlation_id AS "Correlation Id",
# MAGIC                 ticket_id AS "Ticket Id",
# MAGIC                 'Address annotation clustering' AS "Project Type",
# MAGIC                 u.name AS "Agent Name",
# MAGIC                 Option_type AS "Option Type",
# MAGIC                 type AS "Type",
# MAGIC                 u.email_id
# MAGIC             FROM corp_analytics.data_annotation_address_clustering_tbl ma
# MAGIC             LEFT JOIN mjolnir_service.users u ON u.email_id = ma.email_id
# MAGIC             UNION ALL
# MAGIC             SELECT
# MAGIC                 CAST("Annotation Date" AS TIMESTAMP) AS "Annotation Date",
# MAGIC                 "Correlation Id",
# MAGIC                 "Ticket Id",
# MAGIC                 "Project Type",
# MAGIC                 "Agent Name",
# MAGIC                 "Option Type",
# MAGIC                 "Type",
# MAGIC                 email_id
# MAGIC             FROM corp_analytics.data_annotation_spade_template_tagging_v3_tbl
# MAGIC             UNION ALL
# MAGIC             SELECT
# MAGIC                 annotated_at AS "Annotation Date",
# MAGIC                 correlation_id AS "Correlation Id",
# MAGIC                 ticket_id AS "Ticket Id",
# MAGIC                 'SPADE Category Matching V1' AS "Project Type",
# MAGIC                 annotated_by AS "Agent Name",
# MAGIC                 option_type AS "Option Type",
# MAGIC                 type AS "Type",
# MAGIC                 email_id
# MAGIC             FROM corp_analytics.data_annotation_spade_category_v3_tbl ma
# MAGIC             UNION ALL
# MAGIC             SELECT
# MAGIC                 CAST(annotated_at AS TIMESTAMP) AS "Annotation Date",
# MAGIC                 correlation_id AS "Correlation Id",
# MAGIC                 ticket_id AS "Ticket Id",
# MAGIC                 use_case_type AS "Project Type",
# MAGIC                 u.name AS "Agent Name",
# MAGIC                 option_type AS "Option Type",
# MAGIC                 type AS "Type",
# MAGIC                 u.email_id
# MAGIC             FROM corp_analytics.data_annotation_call_annotation_v3_tbl ma
# MAGIC             LEFT JOIN mjolnir_service.users u ON u.email_id = ma.email_id
# MAGIC         ) a
# MAGIC         LEFT JOIN (SELECT ticket_id, td FROM corp_analytics_databricks.data_annotation_online_time_tbl) b
# MAGIC             ON a."Ticket Id" = b.ticket_id
# MAGIC         LEFT JOIN (
# MAGIC             SELECT
# MAGIC                 DATE(starting_t) AS annotation_date,
# MAGIC                 title,
# MAGIC                 ROUND(SUM(td) / COUNT(td), 2) AS average_time,
# MAGIC                 APPROX_PERCENTILE(td, 0.5) AS median_time
# MAGIC             FROM corp_analytics_databricks.data_annotation_online_time_tbl
# MAGIC             GROUP BY 1, 2
# MAGIC         ) c ON DATE(a."Annotation Date") = c.annotation_date AND a."Project Type" = c.title
# MAGIC         LEFT JOIN (
# MAGIC             SELECT
# MAGIC                 DATE(starting_t) AS annotation_date,
# MAGIC                 email_id,
# MAGIC                 title,
# MAGIC                 COUNT(DISTINCT ticket_id) AS project_wise_daily_ticket_counts,
# MAGIC                 SUM(td) AS project_wise_daily_time
# MAGIC             FROM corp_analytics_databricks.data_annotation_online_time_tbl
# MAGIC             GROUP BY 1, 2, 3
# MAGIC         ) d ON DATE(a."Annotation Date") = d.annotation_date AND a.email_id = d.email_id AND a."Project Type" = d.title
# MAGIC         WHERE
# MAGIC             b.td IS NOT NULL
# MAGIC             AND a."Annotation Date" IS NOT NULL
# MAGIC             AND a."Ticket Id" IS NOT NULL
# MAGIC             AND a."Agent Name" IS NOT NULL
# MAGIC     ),
# MAGIC     enriched_data AS (
# MAGIC         SELECT
# MAGIC             *,
# MAGIC             COUNT(*) OVER (PARTITION BY "Correlation Id", "Project Type") AS Ticket_ID_Support,
# MAGIC             CAST(MAX("Annotation Date") OVER (PARTITION BY "Correlation Id", "Project Type") AS DATE) AS "Final Ticket Annotated Date"
# MAGIC         FROM base_table
# MAGIC         WHERE LOWER("Project Type") LIKE '%category matching%'
# MAGIC     )
# MAGIC     SELECT *
# MAGIC     FROM enriched_data
# MAGIC     WHERE "Final Ticket Annotated Date" IN (
# MAGIC             CURRENT_DATE - INTERVAL '1' DAY,
# MAGIC             CURRENT_DATE - INTERVAL '2' DAY
# MAGIC         )
# MAGIC         AND Ticket_ID_Support >= 3
# MAGIC         AND "Type" IN ('Three -', 'Two-One')
# MAGIC         AND RANDOM() < 0.25
# MAGIC ) sub
# MAGIC WHERE 1 = 1
# MAGIC
# MAGIC """
# MAGIC
# MAGIC df = pd.read_sql(query, conn)
# MAGIC display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1b-YX7vZz_ibvmLGIEC_FfpUTIrHeo7HROSDFjIsHnM8/edit?gid=1204962317#gid=1204962317"
worksheet_name = "Master Dump"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)